# ZS601 colored LiDAR initialization: 200 virtual views

This notebook executes zero optimization steps. It saves and reloads an SH0 Gaussian PLY
with opacity 0.999999, renders all fixed near cameras, and evaluates against Blender GT.
Run through the official Colab CLI. Upload the input bundle and source archive first.
The executed notebook, terminal outputs, and hashes are retained. Existing outputs are refused.

See the adjacent README for coordinate, PNG, alpha, scale and license contracts.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, time, platform
import torch
ROOT = Path(os.environ.get('ZS601_RUN_ROOT', '/content/zs601-lidar-init-v001'))
REPO = ROOT/'source'
PKG = REPO/'gaussian-splatting-lidar-init'
INPUT = ROOT/'input'
RUN = ROOT/os.environ.get('ZS601_ATTEMPT', 'run-001')
RUN.mkdir(exist_ok=False)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def save(name,data):
    with (RUN/name).open('x') as f: json.dump(data,f,indent=2,allow_nan=False)
def run(command, log):
    print('$', ' '.join(map(str,command)), flush=True)
    with (RUN/log).open('x') as f:
        proc=subprocess.Popen(list(map(str,command)),cwd=PKG,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            f.write(line); f.flush(); print(line,end='',flush=True)
        rc=proc.wait()
    if rc: raise RuntimeError(f'{log} failed: exit {rc}')
env=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.version.cuda,
         gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0),
         source_commit=os.environ.get('ZS601_CODE_COMMIT'),optimization_steps=0)
save('environment.json',env)
print(json.dumps(env,indent=2))
assert torch.cuda.is_available()
assert env['source_commit'], 'Supply ZS601_CODE_COMMIT from the verified source archive manifest.'


In [ ]:
# Verify immutable source and every input before CUDA build or rendering.
source_manifest=json.loads((ROOT/'source_manifest.json').read_text())
for record in source_manifest['files']:
    assert sha(REPO/record['path']) == record['sha256'], record['path']
assert source_manifest['code_commit'] == env['source_commit']
input_manifest=json.loads((INPUT/'input_manifest.json').read_text())
for record in input_manifest['files']:
    assert sha(INPUT/record['path']) == record['sha256'], record['path']
save('identity_verified.json',dict(code_commit=env['source_commit'],source_files=len(source_manifest['files']),
    input_files=len(input_manifest['files']),point_cloud_sha256=sha(INPUT/'points_3cm.ply')))
print('All source and input hashes verified.')


In [ ]:
# Use the runtime's PyTorch/CUDA; compile pinned repository CUDA sources unchanged.
os.environ['MAX_JOBS']='2'
os.environ['TORCH_CUDA_ARCH_LIST']='.'.join(map(str,torch.cuda.get_device_capability(0)))
run([sys.executable,'-m','pip','install','plyfile==1.1.3','ninja==1.13.0','setuptools==81.0.0','wheel'], 'install_python.log')
DEPS=REPO/'gaussian-splattingWithMask/submodules'
run([sys.executable,'-m','pip','install','--no-build-isolation',str(DEPS/'simple-knn'),str(DEPS/'diff-gaussian-rasterization')], 'build_cuda.log')
import simple_knn._C, diff_gaussian_rasterization
run([sys.executable,'check_contract.py','--sparse',INPUT/'sparse/0'], 'check_contract.log')
save('installed_environment.json',dict(pip_freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),
    nvidia_smi=subprocess.check_output(['nvidia-smi'],text=True)))
print('CUDA extensions imported successfully.')


In [ ]:
# Smoke uses three separated fixed camera IDs; it cannot change initialization settings.
base=[sys.executable,'render_from_sparse_v4.py','--point-cloud',INPUT/'points_3cm.ply','--sparse',INPUT/'sparse/0',
      '--opacity','0.999999','--init-scale-factor','0.5']
smoke=RUN/'smoke'
run(base+['--output',smoke,'--view-ids','3001,3301,3598'], 'smoke_render.log')
verify=[sys.executable,'verify_outputs.py','--point-cloud',INPUT/'points_3cm.ply','--sparse',INPUT/'sparse/0']
run(verify+['--output',smoke,'--expected-views','3'], 'smoke_verify.log')
assert (smoke/'COMPLETE.json').is_file()
print('Three-view smoke, analytic CUDA depth, PNGs, poses and PLY reload passed.')


In [ ]:
# Formal run: same settings and all 200 cameras; no training calls.
formal=RUN/'formal'
run(base+['--output',formal], 'formal_render.log')
run(verify+['--output',formal,'--expected-views','200','--ground-truth',INPUT/'gt'], 'formal_verify.log')
report=json.loads((formal/'verification.json').read_text())
assert report['views']==200 and report['optimization_steps']==0
assert json.loads((smoke/'initialization.json').read_text())['gaussian_ply_sha256'] == json.loads((formal/'initialization.json').read_text())['gaussian_ply_sha256']
save('RUN_COMPLETE.json',dict(status='VERIFIED_INITIALIZATION_NOT_TRAINED',source_commit=env['source_commit'],
    optimization_steps=0,views=report['views'],points=report['points'],smoke_and_formal_same_ply=True))
print(json.dumps(report,indent=2))


## Interpretation

`formal/point_cloud/iteration_0/point_cloud.ply` is the saved Gaussian scene.
`formal/images/` contains 200 native black-background renders. `color/` preserves
straight RGB as in v4; the alpha and masks identify missing cloud coverage.
Metrics compare initialization renderings with synthetic Blender GT only.
They are not a 3DGS training result. The notebook runner saves this actually
executed notebook, including execution counts and outputs, before packaging artifacts.
